SPDX-FileCopyrightText: 2025 Dan J. Bower <dbower@eaps.ethz.ch>

SPDX-License-Identifier: GPL-3.0-or-later

In [ ]:
import logging
from pathlib import Path

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import seaborn as sns

from bedroc import debug_logger
from bedroc.containers import DataContainer
from bedroc.hierarchical import Analyzer, hierarchical_difference_model, zero_difference_model

logger = debug_logger()
logger.setLevel(logging.INFO)

savefig_kwargs = {"dpi": 300, "bbox_inches": "tight", "format": "pdf"}
"""Figure options for savefig"""

# Hierarchical Bayesian analysis for multi-feature data

We apply a hierarchical Bayesian modeling framework to multi-feature data in order to infer systematic differences between two groups, A and B. Each data point contains multiple measured features, and these features may vary widely in scale, noise level, or informativeness. A hierarchical formulation allows the model to borrow statistical strength across features, leading to more stable and interpretable inferences.

In the model, each feature has its own mean in group A and an associated mean difference describing how group B deviates from group A. These per-feature differences are not estimated independently; instead, they are linked through a higher-level distribution controlled by a global scale parameter. This induces partial pooling, which automatically shrinks poorly constrained differences toward zero while allowing genuinely strong signals to stand out.

This approach provides coherent uncertainty quantification, posterior estimates of feature-level effect sizes, and a principled comparison against a simpler zero-difference model. The hierarchical model is therefore well suited for high-dimensional problems where features differ in variability or sample support.

## Generate and plot synthetic data

Set a random seed for reproducibility

In [ ]:
RANDOM_SEED = 123

Load the Zircon data

In [ ]:
filepath: Path = Path(
    "/Users/dan/Documents/academic/projects/zircons/volcanic_plutonic/251105_data/SRMVF Zircon Geochronology and Geochemistry Volcanic PLutonic.xlsx"
)
df: pd.DataFrame = pd.read_excel(filepath, sheet_name="Table S1_SRMVF Zircons")

Process the desired features

In [ ]:
name_columns = ["Sample_name", "Type", "alternate_id"]

# Select features
feature_columns = [
    "Ti_ppm_m49",
    "Hf_ppm_m178",
    "Th_ppm_m232",
    "U_ppm_m238",
    # "Ce_ppm_m140",
    # "Eu_ppm_m151", # not available for plutonic
]
# 2SE uncertainty suffix
std_columns = [f"{feature}_Int2SE" for feature in feature_columns]

df = df[name_columns + feature_columns + std_columns]

# Create a rename mapping only for feature columns
rename_map = {col: f"{col}_feature" for col in feature_columns}

# Apply the rename
df = df.rename(columns=rename_map)

# Drop any rows with missing column data
df = df.dropna()

Some basic filtering of the data and create the data container.

In [ ]:
# Some Ti values for volcanic are abnormally high(?)
filter = True

if filter:
    Ti_max = 200000
    df = df[df["Ti_ppm_m49_feature"] < Ti_max]

    # Some very low Hf values
    Hf_min = 2000
    df = df[df["Hf_ppm_m178_feature"] > Hf_min]

data_container = DataContainer(
    df, data_column="Sample_name", feature_std_suffix="_Int2SE", std_scale=2
)

Define groups.

In [ ]:
df_standardized = data_container.get_dataframe()

# Shuffle the rows
df_shuffled = df_standardized.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Map labels to group idx
group_map = {"plutonic": 0, "volcanic": 1}
group_names = tuple(group_map.keys())
df_shuffled["group_idx"] = df_shuffled["Type"].map(group_map)

# Corner plot
pairgrid = sns.pairplot(
    data_container.df_raw,  # df_standardized,
    hue="Type",
    vars=data_container.feature_columns,
    corner=True,
    plot_kws=dict(alpha=0.4, s=20),
    diag_kws=dict(alpha=0.6),
)

Split the data into training and test sets.

In [ ]:
# 20% test fraction
test_fraction = 0.2

# Compute split index from fraction
n = len(df_shuffled)
split = int(n * (1 - test_fraction))

# Training set
df_train = df_shuffled.iloc[:split]
train_value_np = df_train[data_container.feature_columns].to_numpy()
train_std_np = df_train[data_container.feature_std_columns].to_numpy()
train_group_idx = df_train["group_idx"].to_numpy(dtype=int)

# Test set
df_test = df_shuffled.iloc[split:]
test_value_np = df_test[data_container.feature_columns].to_numpy()
test_std_np = df_test[data_container.feature_std_columns].to_numpy()
test_group_idx = df_test["group_idx"].to_numpy(dtype=int)

This line calls the helper function ``hierarchical_difference_model``, which builds and samples the hierarchical Bayesian model. It returns both the constructed PyMC model object (``model``) and the posterior samples stored in an InferenceData object (``idata``).

In [ ]:
model, idata = hierarchical_difference_model(
    train_value_np, train_group_idx, train_std_np, random_seed=RANDOM_SEED, draws=3000
)

# Analyze the inference

After running the hierarchical Bayesian model and obtaining posterior samples in ``idata``, we create a data analyzer object. This analyzer helps us inspect and interpret the results.

In [ ]:
analyzer = Analyzer(
    model, idata, group_names=group_names, feature_names=data_container.feature_names
)

Use the data analyzer to visualize the results of the hierarchical Bayesian inference.

In [ ]:
ax = analyzer.plot_prior_predictive()
ax.figure.suptitle("Prior Predictive Check")

In [ ]:
ax = analyzer.plot_posterior_predictive(thinning_factor=10)
ax.figure.suptitle("Posterior Predictive Check")

In [ ]:
axes = analyzer.plot_posterior(var_names=["mu_A", "mu_B"])
ax = axes.flatten()[0]
ax.figure.suptitle("Posterior Distributions", fontsize="xx-large")
# Automatically adjust spacing for suptitle
ax.figure.tight_layout(rect=(0, 0, 1, 0.98))  # pyright: ignore
# ax.figure.savefig(f"posterior.{savefig_kwargs['format']}", **savefig_kwargs)  # pyright: ignore

We can compute a Bayesian analogue of feature importance, but that is richer because it shows the full distribution and credible intervals, not just a scalar score.

In [ ]:
axes = analyzer.plot_posterior_differences()
axes[0].set_title(
    f"Posterior Differences ({analyzer.difference_str})", fontdict={"fontsize": "xx-large"}
)

In [ ]:
axes = analyzer.plot_posterior_effect_size()
axes[0].set_title(
    f"Posterior Effect Sizes ({analyzer.difference_str})", fontdict={"fontsize": "xx-large"}
)

For the confusion matrix, we generate some out-of-sample data. The goal is to evaluate the model's classification performance on new data that was not seen during training. This allows us to test generalization, avoiding overly optimistic metrics that could arise if we only measured accuracy on the training set.

Steps:

1. Generate new synthetic samples for both Type A and Type B using the same data generator, but ensuring they are independent of the training data.
2. Stack the new samples together to form a single dataset for prediction.
3. Create the corresponding true labels array to compare against model predictions.
4. Pass this dataset to the analyzer's `plot_confusion_matrix` method to visualize how well the model separates Type A and Type B in previously unseen data.

In [ ]:
ax = analyzer.plot_confusion_matrix(test_value_np, test_group_idx, test_std_np)
ax.set_title(f"Confusion Matrix: {analyzer.difference_str}", fontsize="xx-large")
# ax.figure.savefig(f"confusion_matrix.{savefig_kwargs['format']}", **savefig_kwargs)  # pyright: ignore

## Model comparison

We can compare the hierarchical model with a zero-difference model to assess whether the data support feature-wise mean differences between types A and B. A zero difference model acts as a baseline or null hypothesis: there are no systematic differences between the two groups beyond random noise.

To compare these models, we use Leave-One-Out cross-validation (LOO) based on the pointwise log-likelihood. LOO evaluates how well each model predicts unseen data and automatically penalises model complexity. In simple terms, LOO leaves each data point out, checks how well the model predicts it, and sums up the results.

If the hierarchical model shows a higher Expected Log Predictive Density (ELPD) (less negative LOO), this indicates that the data support feature-level differences between the groups. If not, the simpler zero-difference model may be sufficient.

You may have also heard of Bayes Factors, another approach to model comparison. Bayes factors measure evidence in the data for one model relative to another, but are sensitive to the choice of priors. In contrast, LOO focuses on out-of-sample prediction, which is often more relevant in practice and easier to compute for hierarchical models.

In [ ]:
zero_model, zero_idata = zero_difference_model(
    train_value_np, train_group_idx, random_seed=RANDOM_SEED
)

with model:
    pm.compute_log_likelihood(idata)

with zero_model:
    pm.compute_log_likelihood(zero_idata)

hierarchical_loo = az.loo(idata, var_name="X_obs")
zero_loo = az.loo(zero_idata, var_name="X_obs")

logger.debug("Hierarchical model LOO:\n%s", hierarchical_loo)
logger.debug("Zero-difference model LOO:\n%s", zero_loo)

df_comp_loo = az.compare(
    {"hierarchical": idata, "zero difference": zero_idata}, ic="loo", var_name="X_obs"
)

# Format DataFrame nicely for logging
loo_str = df_comp_loo.to_string(
    float_format="{:.6f}".format,  # round floats to 3 decimals
    justify="right",  # right-align columns
    col_space=10,  # minimum column width
)

logger.info("LOO model comparison:\n%s", loo_str)

There is a convenient function to visualize model comparison results.

In [ ]:
_ = az.plot_compare(df_comp_loo, insample_dev=False)

We also determine which locations are determined the most confidently.

In [ ]:
plutonic_predicted, volcanic_predicted = analyzer.predict_type_posterior(
    test_value_np, test_std_np
)

# Get the location IDs
locations = df_test["alternate_id"].to_numpy()
unique_locs = np.unique(locations)

# Prepare arrays to store mean probabilities per location
p_plutonic_loc = np.zeros(len(unique_locs))
p_volcanic_loc = np.zeros(len(unique_locs))

for i, loc in enumerate(unique_locs):
    idx = np.where(locations == loc)[0]  # all rows corresponding to this location
    # Take mean across both rows and posterior samples
    p_plutonic_loc[i] = plutonic_predicted[idx].mean()
    p_volcanic_loc[i] = volcanic_predicted[idx].mean()

# Build DataFrame
grouped = pd.DataFrame(
    {"location": unique_locs, "p_plutonic": p_plutonic_loc, "p_volcanic": p_volcanic_loc}
)

# Predicted label
grouped["pred_label"] = np.where(
    grouped["p_plutonic"] > grouped["p_volcanic"], "Plutonic", "Volcanic"
)
# True label per location
true_by_location = df_test.groupby("alternate_id")["Type"].first().str.capitalize()

# Correct / incorrect
grouped["is_correct"] = (
    grouped["pred_label"] == true_by_location.loc[grouped["location"]].values
).astype(int)

# Confidence score
grouped["confidence"] = np.abs(grouped["p_plutonic"] - grouped["p_volcanic"])

# Define wrong flag (1 = wrong, 0 = correct)
grouped["wrong"] = 1 - grouped["is_correct"]

conf = grouped["confidence"]
wrong = grouped["wrong"]

grouped["sort_score"] = np.where(wrong == 1, -conf, +conf)

grouped = grouped.sort_values("sort_score").reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(
    2, 1, sharex=True, figsize=(10, 6), gridspec_kw={"height_ratios": [4, 0.5]}
)

# Main bar plot
grouped[["p_plutonic", "p_volcanic"]].plot(kind="bar", ax=ax1)
ax1.set_ylabel("Mean posterior probability")
ax1.set_title(
    "Classification confidence by location:\nOrdered from Confident Misclassifications to Reliable Predictions"
)

# Truth strip
correct_colors = np.where(grouped["is_correct"] == 1, "green", "red")

# Draw correctness bar
ax2.bar(np.arange(len(grouped.index)), np.ones(len(correct_colors)), color=correct_colors)

ax2.set_yticks([])
ax2.set_ylabel("Correct")
ax2.set_xticks(range(len(grouped.index)))
ax2.set_xticklabels(grouped["location"], rotation=90)